# 07 — GRU4Rec trên nhóm timestamp (PyTorch, CPU)

Mô hình neural chính theo tài liệu tổng quan phương pháp: **GRU4Rec tự cài theo bộ tính năng của bản chính thức** (Hidasi & Czapp, RecSys 2023), cộng embedding hành vi.

- Input mỗi bước = trung bình embedding các item (trong vocabulary) của nhóm + trung bình embedding các hành vi của nhóm. Hidden state bắt đầu từ 0 ở mỗi phiên.
- Output bước `t` dự đoán tập item của nhóm `t+1`.
- Loss: **BPR-max** và **cross-entropy** (sampled softmax có hiệu chỉnh log-Q), negative dùng chung `n_sample = 2048` lấy theo `popularity^sample_alpha`; target nhiều item chia nhãn đều `1/|G|`.
- Constrained embedding (dùng chung embedding vào/ra) như bản chính thức.
- Cấu hình xuất phát theo tham số RetailRocket công bố trong repo chính thức: BPR-max, 224 hidden, batch 80, n_sample 2048. Optimizer là Adam/SparseAdam (khác Adagrad của bản gốc) nên learning rate được đặt riêng.
- D3: phiên dài hơn 51 nhóm được cắt thành cửa sổ 51 nhóm chồng 1 nhóm khi train; đánh giá luôn dùng toàn bộ prefix.
- Chọn trên validation, early stopping theo NDCG@20. Seed cố định. **Không đọc test.**
- Đầu ra: `models/gru4rec/{config.json, model.pt}` (không commit `.pt`), `reports/05_08_validation/gru4rec_validation.json`.

## 1. Đường dẫn, giao thức và kiểm tra đầu vào

In [ ]:
from pathlib import Path
import sys, time, json

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, require, utc_now, verify_inputs, write_json
from rr.candidates import CandidateFilter
from rr.samples import prefix_events
from rr.models import popularity, prefix_matrix
from rr.evaluation import load_samples, run, flat
from rr.gru import (GRU4Rec, GroupData, fit, idx_to_itemid, query_states, target_lists, topn_items)

PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
PROCESSED = ROOT / "data/processed"
EVAL_DIR = ROOT / "reports/05_08_validation"
MODEL_DIR = ROOT / "models/gru4rec"
FALLBACK_TOP_N = PROTOCOL["candidates"]["fallback_top_n"]
MAX_LEN = PROTOCOL["long_sessions"]["gru_train_max_prefix_groups"] + 1
SEED = PROTOCOL["seed"]
WIDTH = 200
torch.set_num_threads(4)

samples_report = read_json(ROOT / "reports/04_samples/metrics.json")
metadata_report = read_json(ROOT / "reports/03_metadata/metrics.json")
split_report = read_json(ROOT / "reports/02_session_split/metrics.json")
pop_report = read_json(EVAL_DIR / "popularity_validation.json")
for name, r in [("samples", samples_report), ("popularity_validation", pop_report)]:
    require(r["protocol"]["version"] == PROTOCOL["version"], f"{name} được tạo với protocol khác.")
expected = {p: samples_report["artifacts"][p] for p in [
    "data/processed/item_vocab.parquet", "data/processed/samples/validation_queries.parquet",
    "data/processed/samples/validation_targets.parquet"]}
for p in ["data/processed/item_property_history.parquet", "data/processed/item_catalog.parquet"]:
    expected[p] = metadata_report["artifacts"][p]
expected["data/processed/events_sessionized.parquet"] = split_report["artifacts"]["data/processed/events_sessionized.parquet"]
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
print("Đầu vào khớp SHA-256:", *inputs, sep="\n  ")
print(environment(), "| torch", torch.__version__, "| threads", torch.get_num_threads(), "| protocol", PROTOCOL["version"])

## 2. Ví dụ biên trên dữ liệu nhỏ

- Output bước `t` dự đoán nhóm `t+1` (lệch đúng một bước).
- Hidden state reset giữa các phiên: hai phiên có cùng prefix cho cùng vector.
- Chạy cả phiên một lượt (`sequential`) cho kết quả giống chạy từng prefix riêng (`per_query`): không có thông tin từ nhóm target lọt vào.

In [ ]:
from rr.samples import build_vocab
_ev = pd.DataFrame([(1, 1, 10, "view"), (1, 2, 11, "view"), (1, 2, 12, "addtocart"), (1, 3, 13, "view"),
                    (2, 1, 10, "view"), (2, 2, 99, "view")], columns=["session_id", "group_index", "itemid", "event"])
_vocab = build_vocab(_ev[_ev["itemid"] != 99])
_data = GroupData(_ev, _vocab)
_w = _data.windows(None)
_b, _t, _tgt, _lo, _cnt = target_lists(_data, _w[:, 0], _w[:, 1])
assert _tgt.tolist() == (_w[_b, 0] + _t + 1).tolist() and _cnt.tolist() == [2, 1, 0]
torch.manual_seed(0)
_m = GRU4Rec(int(_vocab["item_idx"].max()) + 1, hidden=8, constrained=True)
_q = pd.DataFrame({"session_id": [1, 1, 2], "prefix_len_groups": [1, 2, 1]})
_seq, _per = query_states(_m, _data, _q), query_states(_m, _data, _q, mode="per_query", batch_size=1)
torch.testing.assert_close(_seq, _per)
torch.testing.assert_close(_seq[0], _seq[2])
del _ev, _vocab, _data, _w, _b, _t, _tgt, _lo, _cnt, _m, _q, _seq, _per
print("Đã kiểm tra: lệch một bước, reset hidden state, sequential = per_query.")

## 3. Dữ liệu, bộ lọc ứng viên và fallback

Số bước dự đoán khi train (tổng `len - 1` trên các cửa sổ) phải bằng đúng số query train của notebook 04.

In [ ]:
vocab = pd.read_parquet(PROCESSED / "item_vocab.parquet")
catalog = pd.read_parquet(PROCESSED / "item_catalog.parquet")
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
events = pd.read_parquet(PROCESSED / "events_sessionized.parquet",
                         columns=["session_id", "group_index", "itemid", "event", "timestamp", "split"])
events["split"] = events["split"].astype(str)
train_events = events[events["split"] == "train"]
val_events = events[events["split"] == "validation"]
flt = CandidateFilter.from_frames(vocab, catalog, history, "known_at_train_ms")
flt_avail = CandidateFilter.from_frames(vocab, catalog, history, "known_at_train_ms", use_availability=True)
fb_items, fb_scores = (a[:FALLBACK_TOP_N] for a in popularity(train_events, pop_report["selected"]["fallback_weights"]))
val_q, val_t = load_samples(ROOT, "validation")

t0 = time.perf_counter()
train_data = GroupData(train_events, vocab)
val_data = GroupData(val_events, vocab)
windows = train_data.windows(MAX_LEN)
n_items = int(vocab["item_idx"].max()) + 1
idx_item = idx_to_itemid(vocab)
train_steps = int((windows[:, 1] - 1).sum())
require(train_steps == samples_report["artifacts"]["data/processed/samples/train_queries.parquet"]["rows"], "Số bước train khác số query train.")
print(f"Chuẩn bị dữ liệu: {time.perf_counter() - t0:.0f}s | cửa sổ train {len(windows):,} | bước dự đoán {train_steps:,} | vocab {n_items:,}")

def evaluate(model, flt_=flt, exclude=None, mode="sequential"):
    t0 = time.perf_counter()
    h = query_states(model, val_data, val_q, mode=mode, seed=SEED)
    items, scores = topn_items(model, h, idx_item, WIDTH)
    q_s = time.perf_counter() - t0
    report, per_q, *_ = run(items, scores, val_q, val_t, flt_, fb_items, fb_scores, exclude=exclude)
    report["query_seconds_total"] = round(q_s, 1)
    report["query_ms_per_query"] = round(1000 * q_s / len(val_q), 4)
    return report

## 4. Smoke test trên khoảng 1% train

Hai epoch ngắn trên 1% cửa sổ train: loss phải giảm.

In [ ]:
rng = np.random.default_rng(SEED)
small = windows[rng.choice(len(windows), len(windows) // 100, replace=False)]
torch.manual_seed(SEED)
smoke = GRU4Rec(n_items, hidden=64, dropout=0.0, constrained=True)
smoke_hist = fit(smoke, train_data, small, epochs=3, lr=0.005, batch_size=80, n_neg=512, seed=SEED, loss="bpr-max")
display(pd.DataFrame(smoke_hist))
require(smoke_hist[-1]["loss"] < smoke_hist[0]["loss"], "Smoke test: loss không giảm.")
del smoke

## 5. Lưới siêu tham số

Giai đoạn 1: so sánh loss ở cấu hình xuất phát chính thức. Giai đoạn 2: với loss tốt nhất, thử hidden nhỏ hơn và dropout embedding nhỏ hơn. Tối đa 8 epoch, dừng sớm khi NDCG@20 validation không tăng 2 epoch liên tiếp.

In [ ]:
OFFICIAL = dict(hidden=224, batch_size=80, n_neg=2048, constrained=True, dropout_embed=0.5, dropout=0.05,
                lr=0.002, sample_alpha=0.5, bpreg=1.0, elu=0.5)
MAX_EPOCHS, PATIENCE = 8, 2
runs = {}

def train_config(name, cfg):
    torch.manual_seed(SEED)
    model = GRU4Rec(n_items, dim=cfg.get("dim", 64), hidden=cfg["hidden"], dropout=cfg["dropout"],
                    dropout_embed=cfg["dropout_embed"], constrained=cfg["constrained"])
    reports = {}
    def on_epoch(epoch):
        reports[epoch] = evaluate(model)
        r = reports[epoch]["all"]
        print(f"  {name} epoch {epoch}: ndcg@20={r['ndcg@20']:.4f} hit@20={r['hit@20']:.4f}", flush=True)
        return r["ndcg@20"]
    t0 = time.perf_counter()
    hist = fit(model, train_data, windows, MAX_EPOCHS, lr=cfg["lr"], batch_size=cfg["batch_size"], n_neg=cfg["n_neg"],
               seed=SEED, on_epoch=on_epoch, patience=PATIENCE, loss=cfg["loss"], bpreg=cfg["bpreg"], elu=cfg["elu"],
               sample_alpha=cfg["sample_alpha"])
    best_epoch = max(reports, key=lambda e: reports[e]["all"]["ndcg@20"])
    runs[name] = {"config": cfg, "history": hist, "best_epoch": best_epoch, "report": reports[best_epoch],
                  "fit_seconds": round(time.perf_counter() - t0, 1)}
    return model

models = {}
for loss in ["bpr-max", "cross-entropy"]:
    models[f"{loss}_official"] = train_config(f"{loss}_official", {**OFFICIAL, "loss": loss})
best_loss = max(["bpr-max", "cross-entropy"], key=lambda l: runs[f"{l}_official"]["report"]["all"]["ndcg@20"])
models = {k: v for k, v in models.items() if k == f"{best_loss}_official"}
for name, change in [("hidden100", {"hidden": 100}), ("dropemb0.25", {"dropout_embed": 0.25})]:
    models[f"{best_loss}_{name}"] = train_config(f"{best_loss}_{name}", {**OFFICIAL, "loss": best_loss, **change})
    best_so_far = max(models, key=lambda n: runs[n]["report"]["all"]["ndcg@20"])
    models = {best_so_far: models[best_so_far]}
best_name = next(iter(models))
best_model = models[best_name]
display(pd.DataFrame({n: {"ndcg@20": r["report"]["all"]["ndcg@20"], "hit@20": r["report"]["all"]["hit@20"],
                          "mrr@20": r["report"]["all"]["mrr@20"], "best_epoch": r["best_epoch"],
                          "fit_min": r["fit_seconds"] / 60} for n, r in runs.items()}).T)
print("Cấu hình tốt nhất:", best_name)

## 6. Ablation cho câu hỏi nghiên cứu 2 và các biến thể báo cáo

- `per_query` phải trùng `sequential` (kiểm tra trên 2.000 query).
- `shuffled`: xáo trộn thứ tự nhóm trong prefix; `last_group`: chỉ dùng nhóm cuối. Nếu thứ tự có ích, hai biến thể này phải thấp hơn.
- `exclude_seen` và biến thể có luật `available` như các notebook trước.

In [ ]:
check_q = val_q.sample(2000, random_state=SEED)
torch.testing.assert_close(query_states(best_model, val_data, check_q),
                           query_states(best_model, val_data, check_q, mode="per_query"), rtol=1e-4, atol=1e-5)
seen = prefix_matrix(prefix_events(val_q, val_events, ("itemid", "event")), val_q["query_id"].to_numpy())
variants = {"full_prefix": runs[best_name]["report"],
            "shuffled_groups": evaluate(best_model, mode="shuffled"),
            "last_group_only": evaluate(best_model, mode="last_group"),
            "exclude_seen": evaluate(best_model, exclude=seen),
            "with_availability": evaluate(best_model, flt_=flt_avail)}
for part in ["all", "new_targets", "repeat_targets", "strong_behavior_targets"]:
    print(part)
    display(flat(variants, part)[["hit@20", "recall@20", "mrr@20", "ndcg@20"]].style.format("{:.4f}"))
display(pd.DataFrame({k: {b: v["ndcg@20"] for b, v in r["by_prefix_len"].items()} for k, r in variants.items()}).style.format("{:.4f}"))

## 7. Lưu mô hình và báo cáo

`best_epoch` của cấu hình tốt nhất là số epoch cố định khi fit lại trên train + validation (D2).

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
config = {"name": best_name, **runs[best_name]["config"], "n_items": n_items, "epochs": runs[best_name]["best_epoch"],
          "seed": SEED, "max_len": MAX_LEN, "vocabulary": "train", "width": WIDTH}
torch.save(best_model.state_dict(), MODEL_DIR / "model.pt")
write_json(MODEL_DIR / "config.json", config)
require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
write_json(EVAL_DIR / "gru4rec_validation.json", {
    "generated_at_utc": utc_now(), **environment(), "torch_version": torch.__version__,
    "notebook": "notebooks/07_gru4rec.ipynb",
    "protocol": {"path": rel(PROTOCOL_PATH, ROOT), "version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
    "inputs": inputs, "split": "validation", "fit_data": "train", "n_queries": len(val_q),
    "fallback": pop_report["selected"]["fallback"], "train_steps": train_steps, "smoke_test": smoke_hist,
    "runs": runs, "selected": config, "model_file": {"path": rel(MODEL_DIR / "model.pt", ROOT), **fingerprint(MODEL_DIR / "model.pt")},
    "variants": variants, "status": "PASS"})
print("Báo cáo:", rel(EVAL_DIR / "gru4rec_validation.json", ROOT))